在很大程度上，Python 生态系统的吸引力来源于这样一个事实：  
几乎所有可用的库都是开源的，并且可以免费使用。

然而，在金融分析，尤其是算法交易中，仅依赖开源软件和算法是不够的——数据同样至关重要。

本书重点介绍了 **Quandl**。  
Quandl 是一个聚合平台，提供大量数据源，包括：
- 开放数据（免费）
- 高级数据（付费）

并且通过统一的 API 提供访问，同时也有 Python 封装库。

### Quandl

In [31]:
# 可以通过 conda 安装 Quandl 的 Python 封装：
# ！pip install quandl

In [32]:
# 读取配置与数据

import configparser

config = configparser.ConfigParser()
config.read('pyalgo.cfg')

['pyalgo.cfg']

In [33]:
# Quandl 的数据请求通常需要指定：数据库名/数据集名
# 例如：BCHAIN/MKPRU 表示比特币价格数据

import quandl as q

data = q.get('BCHAIN/MKPRU', api_key=config['quandl']['api_key'])
data.info()

QuandlError: (Status 403) Something went wrong. Please try again. If you continue to have problems, please contact us at connect@quandl.com.

##### 国内使用 Quandl 的困境

- 网络问题：经常连不上，请求超时。
- API 限制：免费额度很低，很多数据要付费。
- 数据不全：A股数据几乎没有，主要是美股/加密货币。
- 合规 & 授权问题：数据授权经常变化。

##### 国内常见替代方案：

1. 用国内数据源 (稳定，推荐)
- Tushare / yfinance
- JoinQuant
- RiceQuant

2. 自己爬数据 (容易被封 + 不稳定)
- 东方财富
- 新浪财经

3. 用交易所数据 (接口复杂）
- 上交所 / 深交所

### yfinance 替换 Quandl （实现部分功能）

In [34]:
#!pip install yfinance

In [35]:
import yfinance as yf

# BTC/USD
data = yf.download('BTC-USD', start='2009-01-01', end='2020-08-26')

data.head()

/tmp/ipykernel_90008/2703623034.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('BTC-USD', start='2009-01-01', end='2020-08-26')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2014-09-17,457.334015,468.174011,452.421997,465.864014,21056800
2014-09-18,424.440002,456.859985,413.104004,456.859985,34483200
2014-09-19,394.795990,427.834991,384.532013,424.102997,37919700
2014-09-20,408.903992,423.295990,389.882996,394.673004,36863600
2014-09-21,398.821014,412.425995,393.181000,408.084991,26580100


In [36]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2170 entries, 2014-09-17 to 2020-08-25
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Close, BTC-USD)   2170 non-null   float64
 1   (High, BTC-USD)    2170 non-null   float64
 2   (Low, BTC-USD)     2170 non-null   float64
 3   (Open, BTC-USD)    2170 non-null   float64
 4   (Volume, BTC-USD)  2170 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 101.7 KB


In [37]:
yearly = data['Close'].resample('A').last()

yearly

/tmp/ipykernel_90008/3680135079.py:1: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  yearly = data['Close'].resample('A').last()


Ticker,BTC-USD
Date,
2014-12-31,320.192993
2015-12-31,430.566986
2016-12-31,963.742981
2017-12-31,14156.400391
2018-12-31,3742.700439
2019-12-31,7193.599121
2020-12-31,11366.134766


In [38]:
sap = yf.download('SAP', start='2018-01-01', end='2020-05-01')

sap.head()

/tmp/ipykernel_90008/3824829909.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sap = yf.download('SAP', start='2018-01-01', end='2020-05-01')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SAP,SAP,SAP,SAP,SAP
Date,,,,,
2018-01-02,99.060295,99.086736,97.949732,98.037877,540600
2018-01-03,99.871170,100.206105,99.201313,99.271819,506200
2018-01-04,101.404793,101.625142,101.096298,101.166812,539300
2018-01-05,102.532997,102.682833,101.678040,101.757363,475300
2018-01-08,101.184471,101.730933,101.149214,101.440071,515400


In [39]:
sap.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 586 entries, 2018-01-02 to 2020-04-30
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, SAP)   586 non-null    float64
 1   (High, SAP)    586 non-null    float64
 2   (Low, SAP)     586 non-null    float64
 3   (Open, SAP)    586 non-null    float64
 4   (Volume, SAP)  586 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 27.5 KB


In [49]:
# 波动率（替代 VOL/MSFT）
# yfinance 没直接给 implied vol
# 用经典做法：历史波动率

msft = yf.download('MSFT', start='2015-01-01', end='2018-12-31')
returns = msft['Close'].pct_change()
returns = returns.squeeze()

/tmp/ipykernel_90008/2311530299.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  msft = yf.download('MSFT', start='2015-01-01', end='2018-12-31')
[*********************100%***********************]  1 of 1 completed


In [50]:
vol_df = pd.concat([
    returns.rolling(30).std().rename('Vol30'),
    returns.rolling(60).std().rename('Vol60'),
    returns.rolling(90).std().rename('Vol90'),
], axis=1)

vol_df = pd.DataFrame(vol_df)
vol_df.tail()

,Vol30,Vol60,Vol90
Date,,,
2018-12-21,0.021224,0.022649,0.019418
2018-12-24,0.022307,0.023203,0.019846
2018-12-26,0.025860,0.024958,0.021157
2018-12-27,0.025556,0.024925,0.021169
2018-12-28,0.025579,0.024935,0.021173


开源软件是一个已经发展了多年的趋势，它在很多领域（包括算法交易）都降低了进入门槛。与此同时，一个新的、相互促进的趋势是开放数据源的兴起。在某些情况下，比如 Quandl，它们甚至能够提供高质量的数据集。

不过，开放数据在短期内不太可能完全取代专业的付费数据服务，但它们为以较低成本进入算法交易领域提供了一种非常有价值的途径。